In [1]:
import sys
from pathlib import Path

utils_path = Path('/hpc/home/a1/WildfireProject/uavsar-main/python').resolve()
sys.path.append(str(utils_path))

import rasterio
import numpy as np
import matplotlib.pyplot as plt

from process_utils import (
    preprocess_data,
    superpixel_segmentation,
    tv_denoise,
    preprocess_for_merge
)

from rio_utils import reproject_arr_to_match_profile
from scipy.ndimage import generic_filter

In [2]:
def open_one(path):
    with rasterio.open(path) as ds:
        band = ds.read(1)
        profile = ds.profile
    return band, profile

In [3]:
def inc_filter(img):
    img_deg = np.rad2deg(img)
    mask = (img_deg < 20) | (img_deg > 60)
    img_deg[mask] = np.nan
    return img_deg

In [4]:
def apply_incidence_mask(img, inc):
    valid = ~np.isnan(img) & ~np.isnan(inc)
    out = np.full_like(img, np.nan)
    out[valid] = img[valid]
    return out

In [5]:
def fill_nan_with_local_mean(arr, size=9):
    def nanmean_filter(values):
        if np.all(np.isnan(values)):
            return np.nan
        return np.nanmean(values)
    nan_mask = np.isnan(arr)
    filled = generic_filter(arr, nanmean_filter, size=size, mode='nearest')
    arr[nan_mask] = filled[nan_mask]
    return arr

In [6]:
def iterative_fill_nan(arr, size=9, max_iter=10):
    arr = arr.copy()
    for i in range(max_iter):
        nan_before = np.isnan(arr).sum()
        arr = fill_nan_with_local_mean(arr, size=size)
        nan_after = np.isnan(arr).sum()
        if nan_after >= nan_before:
            break
    return arr

In [7]:
data_dir = Path("/hpc/home/a1/FishLakeFireData/ProcessedFiles")
tifs = sorted(list(data_dir.rglob('./*HVHV_rtc_box_crop*.tif')))
tifs

[PosixPath('/hpc/home/a1/FishLakeFireData/ProcessedFiles/fishla_01601_24082_001_241107_HVHV_rtc_box_crop_29.24km_28.02km.tif'),
 PosixPath('/hpc/home/a1/FishLakeFireData/ProcessedFiles/fishla_01601_25031_010_250922_HVHV_rtc_box_crop_29.24km_28.02km.tif')]

In [8]:
data_dir = Path("/hpc/home/a1/FishLakeFireData/ProcessedFiles")
incs = sorted(list(data_dir.rglob('./*inc_box_crop*.tif')))
incs

[PosixPath('/hpc/home/a1/FishLakeFireData/ProcessedFiles/fishla_01601_24082_001_241107_L090_CX_01_inc_box_crop_29.24km_28.02km.tif'),
 PosixPath('/hpc/home/a1/FishLakeFireData/ProcessedFiles/fishla_01601_25031_010_250922_L090_CX_01_inc_box_crop_29.24km_28.02km.tif')]

In [9]:
bands, profiles = zip(*map(open_one, tifs))
bands = list(bands)
for i in range(len(bands)):
    bands[i] = preprocess_for_merge(bands[i])
pre_0 = bands[0]
post_0 = bands[1]
profile_pre0 = profiles[0]
profile_post0 = profiles[1]

In [10]:
inc_bands, inc_profiles = zip(*map(open_one, incs))
inc_bands = list(inc_bands)
for i in range(len(inc_bands)):
    inc_bands[i] = inc_filter(inc_bands[i])
pre_inc_0 = inc_bands[0]
post_inc_0 = inc_bands[1]
profile_pre0_inc = inc_profiles[0]
profile_post0_inc = inc_profiles[1]

In [11]:
resampling = "bilinear"

pre_0, _ = reproject_arr_to_match_profile(pre_0, profile_pre0, profile_pre0, resampling=resampling)
pre_0 = pre_0[0]

post_0, _ = reproject_arr_to_match_profile(post_0, profile_post0, profile_pre0, resampling=resampling)
post_0 = post_0[0]

pre_inc_0, _ = reproject_arr_to_match_profile(pre_inc_0, profile_pre0_inc, profile_pre0, resampling=resampling)
pre_inc_0 = pre_inc_0[0]

post_inc_0, _ = reproject_arr_to_match_profile(post_inc_0, profile_post0_inc, profile_pre0, resampling=resampling)
post_inc_0 = post_inc_0[0]

In [12]:
hv_0 = apply_incidence_mask(pre_0, pre_inc_0)
hv_1 = apply_incidence_mask(post_0, post_inc_0)

In [ ]:
hv_0 = iterative_fill_nan(hv_0, size=9)
hv_1 = iterative_fill_nan(hv_1, size=9)

In [ ]:
print("hv_0 stats:", np.nanmin(hv_0), np.nanmax(hv_0), np.isnan(hv_0).sum())
print("hv_1 stats:", np.nanmin(hv_1), np.nanmax(hv_1), np.isnan(hv_1).sum())
print("valid log10 mask:", np.sum((hv_0 > 0) & (hv_1 > 0)))


In [ ]:
output_path_0 = "/hpc/home/a1/FishLakeFireData/IncMergeFiles/hv_0.tif"
output_path_1 = "/hpc/home/a1/FishLakeFireData/IncMergeFiles/hv_1.tif"

In [ ]:
with rasterio.open(output_path_0, "w", **profile_pre0) as dest:
    dest.write(hv_0, 1)
with rasterio.open(output_path_1, "w", **profile_pre0) as dest:
    dest.write(hv_1, 1)

In [ ]:
plt.imshow(10 * np.log10(hv_0/hv_1))
plt.axis('off')